# Lab 01: GPU 效能驗證與 PyTorch 基本操作

歡迎來到明倫高中暑期 AI 營隊！本單元將引導你完成環境測試，學會如何在 Google Colab 中使用 PyTorch，並驗證顯示卡的算力。

### 任務 1: 匯入 PyTorch 並檢查 GPU 狀態
首先，我們需要匯入 `torch` 套件，並確認目前是否連接了 GPU（雲端 T4 或本地 RTX 4070）。

In [ ]:
import torch
import time

print(f"PyTorch 版本: {torch.__version__}")

# 檢查是否有 GPU (CUDA) 可以使用
gpu_available = torch.cuda.is_available()
print(f"是否可以使用 GPU: {gpu_available}")

if gpu_available:
    print(f"GPU 裝置數量: {torch.cuda.device_count()}")
    print(f"目前使用的 GPU 名稱: {torch.cuda.get_device_name(0)}")
else:
    print("【重要】目前正在使用 CPU，請點選右上角『連線/變更執行階段類型』，將硬體加速器切換為 GPU！")

### 任務 2: 認識 PyTorch 的基本物件：張量 (Tensor)
張量是神經網路計算的基本單位（類似 NumPy 的數組），可以放在 CPU 上運算，也可以放到 GPU 上加速。

請執行下方程式碼，觀察三種張量的建立方式：

In [ ]:
# 1. 建立一個 3x3 的全零張量 (Zeros)
zeros_tensor = torch.zeros(3, 3)
print("全零張量:\n", zeros_tensor)

# 2. 建立一個 2x3 的隨機數值張量 (Random，數值在 0~1 之間)
random_tensor = torch.rand(2, 3)
print("隨機張量:\n", random_tensor)

# 3. 從 Python 列表建立張量
list_data = [[1, 2, 3], [4, 5, 6]]
custom_tensor = torch.tensor(list_data)
print("自訂張量:\n", custom_tensor)
print("張量的形狀 (Shape):", custom_tensor.shape)

### 任務 3: 將張量移動到 GPU (CUDA)
預設建立的張量是在 CPU 記憶體中，我們要學會如何將它移至 GPU 上。

In [ ]:
# 定義運算裝置 (Device)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"目前使用的運算裝置是: {device}")

# 建立一個隨機張量
x = torch.rand(3, 3)
print("x 的原始儲存位置 (預設在 CPU):", x.device)

In [ ]:
# TODO: 請使用 .to(device) 方法將張量 x 移至 GPU 執行階段
# x_gpu = 

# 檢查移轉後的儲存位置 (完成填空後請將下方 print 註解拿掉)
# print("x_gpu 的儲存位置:", x_gpu.device)

### 任務 3.5: 將張量由 GPU 移回 CPU
在神經網路模型運算完畢後，有時我們需要將資料移回 CPU 記憶體，以便轉為 NumPy 格式進行繪圖或輸出（因為 NumPy 套件並不支援直接讀取 GPU 上的張量）。

我們可以使用 `.cpu()` 方法或 `.to("cpu")` 方法，將張量從 GPU 轉移回 CPU 裝置。

In [ ]:
# TODO: 請使用 .cpu() 或 .to("cpu") 方法，將 x_gpu 移回 CPU，並指派給變數 x_cpu
# x_cpu = 

# 檢查移回 CPU 後的儲存位置 (完成填空後請將下方 print 註解拿掉)
# print("x_cpu 的儲存位置:", x_cpu.device)

### 任務 4: 運算速度大對決 (CPU vs GPU)
我們讓 CPU 與 GPU 做一次矩陣乘法的大型運算（例如 5000 x 5000 的矩陣），比較兩者所需的時間差距！

In [ ]:
size = 5000

# 1. CPU 矩陣乘法運算
print(f"正在 CPU 上進行 {size}x{size} 矩陣乘法運算...")
mat_a_cpu = torch.rand(size, size)
mat_b_cpu = torch.rand(size, size)

start_time = time.time()
result_cpu = torch.matmul(mat_a_cpu, mat_b_cpu)
cpu_time = time.time() - start_time
print(f"CPU 運算完成！耗時: {cpu_time:.4f} 秒")

# 2. GPU (CUDA) 矩陣乘法運算
if torch.cuda.is_available():
    print(f"正在 GPU 上進行 {size}x{size} 矩陣乘法運算...")
    # 將矩陣移到 GPU 上
    mat_a_gpu = mat_a_cpu.to(device)
    mat_b_gpu = mat_b_cpu.to(device)
    
    # GPU 第一次運算需要預熱，先跑一次小運算
    _ = torch.matmul(mat_a_gpu[:10, :10], mat_b_gpu[:10, :10])
    torch.cuda.synchronize() # 確保前置工作完成
    
    start_time = time.time()
    result_gpu = torch.matmul(mat_a_gpu, mat_b_gpu)
    torch.cuda.synchronize() # 確保 GPU 運算結束再計時
    gpu_time = time.time() - start_time
    print(f"GPU 運算完成！耗時: {gpu_time:.4f} 秒")
    
    # 計算加速倍數
    speedup = cpu_time / gpu_time
    print(f"=======================================")
    print(f"GPU 的運算速度是 CPU 的 {speedup:.1f} 倍！")
    print(f"=======================================")
else:
    print("未偵測到 GPU，無法進行對決測試。")

### 自我挑戰課後題
請嘗試在下方建立一個大小為 2000 x 2000 的全 1 張量，並將其放置於 GPU 上，然後輸出其裝置資訊。

In [ ]:
# 課後挑戰 (選做)：建立一個 2000x2000 的全 1 張量並放到 GPU 上
# 提示：全 1 張量用 torch.ones(列數, 行數)
my_tensor = torch.ones(2000, 2000).to(device)
print("挑戰題張量 shape:", my_tensor.shape)
print("挑戰題張量 device:", my_tensor.device)

---

## Lab 01 學習重點小結
### 核心概念掌握
- PyTorch 張量 (Tensor) 是多維數值陣列，是深度學習的基本資料單位。
- 使用 `torch.cuda.is_available()` 可確認 GPU 是否就緒。
- `.to(device)` 可將張量從 CPU 搬移至 GPU，大幅加速矩陣運算。
- GPU 加速矩陣乘法的核心原理是**平行計算**，數千個核心同時運作。

### 快速自評測驗

**Q1. 下列何者是正確的 2D 張量宣告？**
- A. `torch.tensor(5.0)`
- B. `torch.zeros(3, 4)`
- C. `torch.ones([3])`

<details><summary>查看解答</summary>

**答案：B — `torch.zeros(3, 4)` 建立一個 3 列 4 行的二維零矩陣。**

</details>

**Q2. 若 GPU 不可用，`device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')` 的結果為？**
- A. 報錯
- B. device = 'cpu'
- C. device = None

<details><summary>查看解答</summary>

**答案：B — 條件為 False 時，device 設為 cpu，程式可繼續執行。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「PyTorch 張量 (Tensor) 是多維數值陣列，是深」
- [ ] 我可以用一句話解釋「使用 `torch.cuda.is_available()`」
- [ ] 我可以用一句話解釋「`.to(device)` 可將張量從 CPU 搬移至 GP」
- [ ] 我的程式碼成功執行並得到預期輸出
